# BNMP — Extração bronze (API do BNMP 2.0)

Coleta os dados do Banco Nacional de Monitoramento de Prisões (BNMP 2.0,
PDPJ/CNJ) e grava os JSON brutos no Lakehouse `mp_bronze`, seção Files, sob
`bnmp/json/`, usando `python/src/modulos/bnmp/` do repositório
`mpsp-jurimetria/proj202607`.

O que é gravado:
- `bnmp/json/contexto_sessao.json`, `dominios.json`, `status_pessoas.json`
- `bnmp/json/{recurso}/{consulta}/pagina_NNNNNN.json` — uma página por arquivo
- `bnmp/json/{recurso}/{consulta}/_manifesto.json` — progresso da coleta

**Pré-requisitos:**
- Segredos no Key Vault `KV-Jurimetria`: `BNMP-USUARIO`, `BNMP-SENHA` e
  `BNMP-OTP-SECRET` (segredo base32 do app autenticador).
- O 2º fator já deve estar **cadastrado** na conta do BNMP. Se o SSO ainda
  pedir o cadastro (required action `CONFIGURE_TOTP`), conclua uma vez pelo
  navegador guardando o segredo base32 — enquanto isso não for feito, o
  Keycloak gera um segredo novo a cada login e a automação falha com essa
  mensagem.
- Identidade do notebook com leitura no Key Vault e escrita no `mp_bronze`.

A coleta é **retomável**: reexecutar a célula de execução pula as páginas já
gravadas no Lakehouse. Coletas longas (milhões de registros) sobrevivem à
expiração da sessão do SSO (~8h) — o cliente refaz o login sozinho.

In [ ]:
%pip install --quiet git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

Os IDs do Fabric não são segredos, mas evite deixar valores reais commitados
aqui. As credenciais do BNMP vêm sempre do Key Vault.

In [ ]:
import os

from notebookutils import credentials

KV_URI = "https://KV-Jurimetria.vault.azure.net"
os.environ["BNMP_USER"] = credentials.getSecret(KV_URI, "BNMP-USUARIO")
os.environ["BNMP_PASSWORD"] = credentials.getSecret(KV_URI, "BNMP-SENHA")
os.environ["BNMP_OTP_SECRET"] = credentials.getSecret(KV_URI, "BNMP-OTP-SECRET")

# 39 = Ministério Público do Estado de São Paulo
os.environ["BNMP_ORGAO_ATIVO"] = "39"

os.environ["FABRIC_WORKSPACE_ID"] = "<id do workspace>"
os.environ["FABRIC_LAKEHOUSE_ID"] = "<id do lakehouse mp_bronze>"

## Execução

Os filtros são parametrizáveis. Recomenda-se **quebrar a coleta por UF** (e,
em UF grande, por status) em vez de uma única consulta nacional: a paginação
por offset degrada em consultas de milhões de registros, e o recorte isola
falhas e permite retomada mais granular.

Antes da primeira coleta grande, rode `python/scripts/explorar_api_bnmp.py`
para medir o tamanho de página aceito pela API e o limite de paginação
profunda.

In [ ]:
from src.modulos.bnmp.etl.extract_bronze import executar
from src.modulos.bnmp.filtros import filtro_pessoas

executar(
    consultas_pessoas=[
        ("pessoas_uf-26_ativo-1", filtro_pessoas(uf_id=26)),
    ],
    tamanho_pagina=200,
)

## Verificação

In [ ]:
from src.modulos.bnmp.etl import read_bronze

dominios = read_bronze.ler_dominios()
print(f"dominios: {len(dominios)} listas de referência")

manifesto = read_bronze.ler_manifesto("pessoas", "pessoas_uf-26_ativo-1")
print(manifesto["status"], manifesto["paginas_gravadas"], "de", manifesto["total_paginas"], "páginas")
print(manifesto["total_elementos"], "elementos no total")

envelope = read_bronze.ler_pagina("pessoas", "pessoas_uf-26_ativo-1", 0)
print("1º registro:", envelope["resposta"]["content"][0]["dadosGeraisPessoa"]["nome"])